# Manual DAG benchmark: CD / GOLEM / GES / SP

本 Notebook 使用 `experiments/test_manual_B_Omega_cd_algorithms.py` 中的 9 组人工 DAG 数据生成方式，
并参考 `experiments/validate_er_graph_cd.ipynb` 的参数风格与评估方式。

目标算法：
- `cd_A_l0_0.0`, `cd_A_l0_0.1`, `cd_A_l0_0.2`
- `cd_BOmega_l0_0.0`, `cd_BOmega_l0_0.1`, `cd_BOmega_l0_0.2`
- `cd_B_l0_0.0`, `cd_B_l0_0.1`, `cd_B_l0_0.2`
- `ges`, `golem_ev`, `golem_nv`, `sp`

每个 experiment 独立生成 50 次数据（可配），并统计：
- `mec_match_mean`
- `exact_match_mean`
- `cpdag_shd_mean`
- `cpdag_shd_std`
- `runtime_sec_mean`

In [1]:
import os
import sys
import time
import json
import platform
from datetime import datetime
from itertools import permutations

import numpy as np
import pandas as pd

print('Python :', sys.version.split()[0])
print('Platform:', platform.platform())
print('numpy  :', np.__version__)
print('pandas :', pd.__version__)

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
print('Repo root:', repo_root)

from MEC import is_in_markov_equiv_class, get_skeleton, find_v_structures
from SCM_data import generate_scm_from_BN
from coordinate_descent.coordinate0 import dag_coordinate_descent_l0_epoch as cd_A
from coordinate_descent.cd_B import dag_coordinate_descent_B_epoch as cd_B
from coordinate_descent.cd_B_Omega import dag_coordinate_descent_BOmega_epoch as cd_BOmega

GOLEM_IMPORT_ERROR = None
try:
    golem_src = os.path.join(repo_root, 'golemMain', 'src')
    if golem_src not in sys.path:
        sys.path.append(golem_src)
    from golem import golem as golem_fit
    HAS_GOLEM = True
except Exception as e:
    HAS_GOLEM = False
    GOLEM_IMPORT_ERROR = e
    print('GOLEM unavailable, will skip GOLEM-EV/NV:', e)

GES_IMPORT_ERROR = None
try:
    from causallearn.search.ScoreBased.GES import ges as ges_fit
    HAS_GES = True
except Exception as e:
    HAS_GES = False
    GES_IMPORT_ERROR = e
    print('GES unavailable, will skip GES:', e)

try:
    if os.path.join(repo_root, 'toolbox') not in sys.path:
        sys.path.append(os.path.join(repo_root, 'toolbox'))
    from cdt.metrics import SHD_CPDAG as _SHD_CPDAG
    HAS_CPDAG_SHD = True
except Exception as e:
    _SHD_CPDAG = None
    HAS_CPDAG_SHD = False
    print('CPDAG-SHD backend unavailable, will use fallback:', e)


CFG = {
    'n_samples': 10000,
    'n_repeats': 50,
    'master_seed': 42,
    'threshold': 0.05,
    'k': None,
    'dag_tol': 1e-8,
    'epochs_a': 500,
    'epochs_b': 500,
    'epochs_bomega': 500,
    'lambda_l0_list': [0.0, 0.1, 0.2],
    'tol': 1e-4,
    'patience': 50,
    'min_epochs': 100,
    'eps_omega': 1e-8,
    'run_sp': True,
    'run_ges': True,
    'run_golem': True,
    'golem_num_iter': 20000,
    'golem_learning_rate': 1e-3,
    'golem_lambda1_ev': 2e-2,
    'golem_lambda1_nv': 2e-3,
    'golem_lambda2': 5.0,
    'out_dir': os.path.join(repo_root, 'experiments', 'results'),
    'tag': 'manual_dag_validate_all_algorithms',
}
os.makedirs(CFG['out_dir'], exist_ok=True)

ALGORITHM_ORDER = [
    'cd_A_l0_0.0',
    'cd_A_l0_0.1',
    'cd_A_l0_0.2',
    'cd_BOmega_l0_0.0',
    'cd_BOmega_l0_0.1',
    'cd_BOmega_l0_0.2',
    'cd_B_l0_0.0',
    'cd_B_l0_0.1',
    'cd_B_l0_0.2',
    'ges',
    'golem_ev',
    'golem_nv',
    'sp',
]


def get_manual_experiments():
    exps = []
    exps.append({'experiment_id': 1, 'name': 'd=3, A→B←C', 'B_true': np.array([[0,1,0],[0,0,0],[0,2,0]], dtype=float), 'Omega_true': np.diag([1,2,3]).astype(float)})
    exps.append({'experiment_id': 2, 'name': 'd=3, A→B→C', 'B_true': np.array([[0,1,0],[0,0,3],[0,0,0]], dtype=float), 'Omega_true': np.diag([1,3,4]).astype(float)})
    exps.append({'experiment_id': 3, 'name': 'd=3, A→B→C + A→C', 'B_true': np.array([[0,1,2],[0,0,3],[0,0,0]], dtype=float), 'Omega_true': np.diag([5,4,3]).astype(float)})
    exps.append({'experiment_id': 4, 'name': 'd=4, A→B, B→C, B→D', 'B_true': np.array([[0,2,0,0],[0,0,3,4],[0,0,0,0],[0,0,0,0]], dtype=float), 'Omega_true': np.diag([1,4,3,2]).astype(float)})
    exps.append({'experiment_id': 5, 'name': 'd=4, A→C, A→D, B→C, B→D', 'B_true': np.array([[0,0,2,3],[0,0,3,4],[0,0,0,0],[0,0,0,0]], dtype=float), 'Omega_true': np.diag([2,4,3,5]).astype(float)})
    exps.append({'experiment_id': 6, 'name': 'd=4, A→D, B→D, C→D', 'B_true': np.array([[0,0,0,1],[0,0,0,3],[0,0,0,5],[0,0,0,0]], dtype=float), 'Omega_true': np.diag([5,4,3,2]).astype(float)})
    exps.append({'experiment_id': 7, 'name': 'd=5, e=4, |v|=0', 'B_true': np.array([[0,1,0,2,0],[0,0,3,0,4],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=float), 'Omega_true': np.diag([1,2,3,2,1]).astype(float)})
    exps.append({'experiment_id': 8, 'name': 'd=5, e=4, |v|=1', 'B_true': np.array([[0,0,1,2,0],[0,0,0,2,3],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=float), 'Omega_true': np.diag([1,2,3,2,1]).astype(float)})
    exps.append({'experiment_id': 9, 'name': 'd=5, e=4, |v|=2', 'B_true': np.array([[0,0,0,1,0],[0,0,0,2,3],[0,0,0,0,4],[0,0,0,0,0],[0,0,0,0,0]], dtype=float), 'Omega_true': np.diag([1,2,3,2,1]).astype(float)})
    return exps


def weight_to_binary_adj(W: np.ndarray, threshold: float) -> np.ndarray:
    G = (np.abs(W) > threshold).astype(int)
    np.fill_diagonal(G, 0)
    return G


def cpdag_shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    if HAS_CPDAG_SHD:
        try:
            return float(_SHD_CPDAG(G_true.astype(int), G_est.astype(int)))
        except Exception:
            pass
    skel_true = get_skeleton(G_true)
    skel_est = get_skeleton(G_est)
    skeleton_diff = int(np.sum(np.abs(skel_true - skel_est)) // 2)
    v_true = find_v_structures(G_true)
    v_est = find_v_structures(G_est)
    v_diff = len(v_true.symmetric_difference(v_est))
    return float(skeleton_diff + v_diff)


def evaluate_algorithm(G_true: np.ndarray, G_est: np.ndarray) -> dict:
    return {
        'mec_match': int(is_in_markov_equiv_class(G_true, G_est)),
        'exact_match': int(np.array_equal(G_true, G_est)),
        'cpdag_shd': cpdag_shd_score(G_true, G_est),
    }


def sp_estimate_W(X: np.ndarray) -> np.ndarray:
    from numpy.linalg import inv, cholesky, LinAlgError
    n, p = X.shape
    if p > 8:
        raise ValueError(f'SP exhaustive search is too expensive for d={p}; please use d<=8.')

    Sigma_hat = np.cov(X, rowvar=False)

    def l0_norm(U, threshold=0.05):
        return int(np.sum(np.abs(U) > threshold))

    best_score = np.inf
    best_W, best_P = None, None

    for perm in permutations(range(p)):
        P = np.eye(p)[list(perm)]
        Sigma_perm = P @ Sigma_hat @ P.T
        try:
            Theta = inv(Sigma_perm)
            L = cholesky(Theta)
            diag_L = np.diag(L)
            sqrt_Omega = np.diag(1.0 / diag_L)
            W = np.eye(p) - L @ sqrt_Omega
            score = l0_norm(W)
            if score < best_score:
                best_score = score
                best_W = W
                best_P = P
        except LinAlgError:
            continue

    if best_W is None:
        raise RuntimeError('SP failed to find a valid structure.')

    W_est = best_P.T @ best_W @ best_P
    np.fill_diagonal(W_est, 0.0)
    return W_est


def ges_graph_to_adj(g: np.ndarray) -> np.ndarray:
    g = np.asarray(g)
    d = g.shape[0]
    A = np.zeros((d, d), dtype=int)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = g[i, j], g[j, i]
            if a == -1 and b == 1:
                A[i, j] = 1
            elif a == 1 and b == -1:
                A[j, i] = 1
            elif a == -1 and b == -1:
                A[i, j] = 1
                A[j, i] = 1
            elif a != 0 or b != 0:
                A[i, j] = int(a != 0)
                A[j, i] = int(b != 0)
    np.fill_diagonal(A, 0)
    return A


def _append_row(rows, exp_id, exp_name, repeat_id, seed, alg, runtime_sec, G_true, G_est):
    m = evaluate_algorithm(G_true, G_est)
    rows.append({
        'experiment_id': int(exp_id),
        'experiment': exp_name,
        'repeat_id': int(repeat_id),
        'seed': int(seed),
        'algorithm': alg,
        'mec_match': m['mec_match'],
        'exact_match': m['exact_match'],
        'cpdag_shd': m['cpdag_shd'],
        'runtime_sec': float(runtime_sec),
    })


def run_manual_dag_benchmark(cfg: dict, n_repeats: int = 50, save_tag: str = None):
    if save_tag is None:
        save_tag = cfg['tag']

    exps = get_manual_experiments()
    rng = np.random.default_rng(cfg['master_seed'])
    rows = []
    skip_logs = []

    for exp in exps:
        exp_id = exp['experiment_id']
        exp_name = exp['name']
        B_true = np.asarray(exp['B_true'], dtype=float)
        Omega_true = np.asarray(exp['Omega_true'], dtype=float)
        N = np.diag(Omega_true).copy()
        d = B_true.shape[0]
        seeds = rng.integers(0, 10**9, size=n_repeats)

        for r_idx, seed in enumerate(seeds, start=1):
            X, _, _, _ = generate_scm_from_BN(B_true.T, n_samples=cfg['n_samples'], N=N, seed=int(seed))
            S = X.T @ X / X.shape[0]
            G_true = weight_to_binary_adj(B_true, threshold=0.0)

            for lam in cfg['lambda_l0_list']:
                lam_tag = f'{lam:.1f}'

                t0 = time.perf_counter()
                _, G_A, _, _ = cd_A(
                    S=S,
                    n_epochs=cfg['epochs_a'],
                    seed=int(seed),
                    threshold=cfg['threshold'],
                    lambda_l0=lam,
                    tol=cfg['tol'],
                    patience=cfg['patience'],
                    min_epochs=cfg['min_epochs'],
                    verbose=False,
                )
                t1 = time.perf_counter()
                _append_row(rows, exp_id, exp_name, r_idx, seed, f'cd_A_l0_{lam_tag}', t1 - t0, G_true, G_A)

                t0 = time.perf_counter()
                _, G_B, _, _, _ = cd_B(
                    S=S,
                    n_epochs=cfg['epochs_b'],
                    seed=int(seed),
                    threshold=cfg['threshold'],
                    lambda_l0=lam,
                    k=cfg['k'],
                    dag_tol=cfg['dag_tol'],
                    tol=cfg['tol'],
                    patience=cfg['patience'],
                    min_epochs=cfg['min_epochs'],
                    verbose=False,
                )
                t1 = time.perf_counter()
                _append_row(rows, exp_id, exp_name, r_idx, seed, f'cd_B_l0_{lam_tag}', t1 - t0, G_true, G_B)

                t0 = time.perf_counter()
                _, G_BOm, _, _, _ = cd_BOmega(
                    S=S,
                    Omega=np.eye(d),
                    n_epochs=cfg['epochs_bomega'],
                    seed=int(seed),
                    threshold=cfg['threshold'],
                    lambda_l0=lam,
                    k=cfg['k'],
                    dag_tol=cfg['dag_tol'],
                    tol=cfg['tol'],
                    patience=cfg['patience'],
                    min_epochs=cfg['min_epochs'],
                    eps_omega=cfg['eps_omega'],
                    verbose=False,
                )
                t1 = time.perf_counter()
                _append_row(rows, exp_id, exp_name, r_idx, seed, f'cd_BOmega_l0_{lam_tag}', t1 - t0, G_true, G_BOm)

            if cfg.get('run_golem', True) and HAS_GOLEM:
                try:
                    t0 = time.perf_counter()
                    B_ev = golem_fit(
                        X,
                        lambda_1=cfg['golem_lambda1_ev'],
                        lambda_2=cfg['golem_lambda2'],
                        equal_variances=True,
                        num_iter=cfg['golem_num_iter'],
                        learning_rate=cfg['golem_learning_rate'],
                        seed=int(seed),
                    )
                    t1 = time.perf_counter()
                    G_ev = weight_to_binary_adj(B_ev, threshold=cfg['threshold'])
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'golem_ev', t1 - t0, G_true, G_ev)

                    t0 = time.perf_counter()
                    B_nv = golem_fit(
                        X,
                        lambda_1=cfg['golem_lambda1_nv'],
                        lambda_2=cfg['golem_lambda2'],
                        equal_variances=False,
                        num_iter=cfg['golem_num_iter'],
                        learning_rate=cfg['golem_learning_rate'],
                        seed=int(seed),
                    )
                    t1 = time.perf_counter()
                    G_nv = weight_to_binary_adj(B_nv, threshold=cfg['threshold'])
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'golem_nv', t1 - t0, G_true, G_nv)
                except Exception as e:
                    skip_logs.append({'experiment_id': exp_id, 'repeat_id': r_idx, 'seed': int(seed), 'algorithm': 'golem_ev/golem_nv', 'reason': str(e)})

            if cfg.get('run_sp', True):
                try:
                    t0 = time.perf_counter()
                    B_sp = sp_estimate_W(X)
                    t1 = time.perf_counter()
                    G_sp = weight_to_binary_adj(B_sp, threshold=cfg['threshold'])
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'sp', t1 - t0, G_true, G_sp)
                except Exception as e:
                    skip_logs.append({'experiment_id': exp_id, 'repeat_id': r_idx, 'seed': int(seed), 'algorithm': 'sp', 'reason': str(e)})

            if cfg.get('run_ges', True) and HAS_GES:
                try:
                    t0 = time.perf_counter()
                    rec = ges_fit(X)
                    t1 = time.perf_counter()
                    G_ges = ges_graph_to_adj(rec['G'].graph)
                    _append_row(rows, exp_id, exp_name, r_idx, seed, 'ges', t1 - t0, G_true, G_ges)
                except Exception as e:
                    skip_logs.append({'experiment_id': exp_id, 'repeat_id': r_idx, 'seed': int(seed), 'algorithm': 'ges', 'reason': str(e)})

        print(f"[Experiment {exp_id}/9] {exp_name} done")

    df_trials = pd.DataFrame(rows)
    df_summary = (
        df_trials
        .groupby(['experiment_id', 'experiment', 'algorithm'], as_index=False)
        .agg(
            mec_match_mean=('mec_match', 'mean'),
            exact_match_mean=('exact_match', 'mean'),
            cpdag_shd_mean=('cpdag_shd', 'mean'),
            cpdag_shd_std=('cpdag_shd', 'std'),
            runtime_sec_mean=('runtime_sec', 'mean'),
            runs=('algorithm', 'size'),
        )
    )

    full_index = pd.MultiIndex.from_product(
        [
            sorted(df_trials['experiment_id'].unique()) if len(df_trials) > 0 else list(range(1, 10)),
            ALGORITHM_ORDER,
        ],
        names=['experiment_id', 'algorithm']
    )

    if len(df_summary) > 0:
        exp_name_map = df_summary[['experiment_id', 'experiment']].drop_duplicates().set_index('experiment_id')['experiment'].to_dict()
        df_summary = df_summary.set_index(['experiment_id', 'algorithm']).reindex(full_index).reset_index()
        df_summary['experiment'] = df_summary['experiment_id'].map(exp_name_map)
        df_summary = df_summary[['experiment_id', 'experiment', 'algorithm', 'mec_match_mean', 'exact_match_mean', 'cpdag_shd_mean', 'cpdag_shd_std', 'runtime_sec_mean', 'runs']]
    else:
        df_summary = pd.DataFrame(columns=['experiment_id', 'experiment', 'algorithm', 'mec_match_mean', 'exact_match_mean', 'cpdag_shd_mean', 'cpdag_shd_std', 'runtime_sec_mean', 'runs'])

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    trials_path = os.path.join(cfg['out_dir'], f'{save_tag}_trials_{ts}.csv')
    summary_path = os.path.join(cfg['out_dir'], f'{save_tag}_summary_{ts}.csv')
    trials_latest_path = os.path.join(cfg['out_dir'], f'{save_tag}_trials.csv')
    summary_latest_path = os.path.join(cfg['out_dir'], f'{save_tag}_summary.csv')
    skips_path = os.path.join(cfg['out_dir'], f'{save_tag}_skips_{ts}.json')

    df_trials.to_csv(trials_path, index=False)
    df_summary.to_csv(summary_path, index=False)
    df_trials.to_csv(trials_latest_path, index=False)
    df_summary.to_csv(summary_latest_path, index=False)
    with open(skips_path, 'w', encoding='utf-8') as f:
        json.dump(skip_logs, f, ensure_ascii=False, indent=2)

    print('HAS_GOLEM:', HAS_GOLEM)
    print('HAS_GES  :', HAS_GES)
    print('HAS_CPDAG_SHD:', HAS_CPDAG_SHD)
    print('Trials saved :', trials_path)
    print('Summary saved:', summary_path)
    print('Skips saved  :', skips_path)

    return df_trials, df_summary, skip_logs


print('Config ready. n_repeats =', CFG['n_repeats'])
print('Algorithms:', ALGORITHM_ORDER)

Python : 3.12.3
Platform: Windows-11-10.0.26200-SP0
numpy  : 1.26.4
pandas : 2.2.2
Repo root: c:\Users\super\DAG
c:\Users\super\DAG\experiments



No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


Config ready. n_repeats = 50
Algorithms: ['cd_A_l0_0.0', 'cd_A_l0_0.1', 'cd_A_l0_0.2', 'cd_BOmega_l0_0.0', 'cd_BOmega_l0_0.1', 'cd_BOmega_l0_0.2', 'cd_B_l0_0.0', 'cd_B_l0_0.1', 'cd_B_l0_0.2', 'ges', 'golem_ev', 'golem_nv', 'sp']


In [2]:
# 先做一次快速 smoke run（每个 experiment 1 次）验证流程
# 为保证快速验证，这里关闭 GOLEM；正式实验单元仍按 CFG 原设置运行。
CFG_SMOKE = dict(CFG)
CFG_SMOKE['n_repeats'] = 1
CFG_SMOKE['run_golem'] = False
CFG_SMOKE['tag'] = CFG['tag'] + '_smoke'

df_trials_smoke, df_summary_smoke, skip_logs_smoke = run_manual_dag_benchmark(
    cfg=CFG_SMOKE,
    n_repeats=CFG_SMOKE['n_repeats'],
    save_tag=CFG_SMOKE['tag'],
)

print('Smoke trials shape:', df_trials_smoke.shape)
print('Smoke skip count:', len(skip_logs_smoke))
display(df_summary_smoke.head(30))

[Experiment 1/9] d=3, A→B←C done
[Experiment 2/9] d=3, A→B→C done
[Experiment 3/9] d=3, A→B→C + A→C done
[Experiment 4/9] d=4, A→B, B→C, B→D done
[Experiment 5/9] d=4, A→C, A→D, B→C, B→D done
[Experiment 6/9] d=4, A→D, B→D, C→D done
[Experiment 7/9] d=5, e=4, |v|=0 done
[Experiment 8/9] d=5, e=4, |v|=1 done
[Experiment 9/9] d=5, e=4, |v|=2 done
HAS_GOLEM: True
HAS_GES  : True
HAS_CPDAG_SHD: True
Trials saved : c:\Users\super\DAG\experiments\results\manual_dag_validate_all_algorithms_smoke_trials_20260219_124725.csv
Summary saved: c:\Users\super\DAG\experiments\results\manual_dag_validate_all_algorithms_smoke_summary_20260219_124725.csv
Skips saved  : c:\Users\super\DAG\experiments\results\manual_dag_validate_all_algorithms_smoke_skips_20260219_124725.json
Smoke trials shape: (99, 9)
Smoke skip count: 0


,experiment_id,experiment,algorithm,mec_match_mean,exact_match_mean,cpdag_shd_mean,cpdag_shd_std,runtime_sec_mean,runs
0,1,"d=3, A→B←C",cd_A_l0_0.0,1.0,1.0,0.0,NaN,0.969313,1.0
1,1,"d=3, A→B←C",cd_A_l0_0.1,1.0,1.0,0.0,NaN,0.447136,1.0
2,1,"d=3, A→B←C",cd_A_l0_0.2,1.0,1.0,0.0,NaN,0.407539,1.0
3,1,"d=3, A→B←C",cd_BOmega_l0_0.0,1.0,1.0,0.0,NaN,0.367327,1.0
4,1,"d=3, A→B←C",cd_BOmega_l0_0.1,1.0,1.0,0.0,NaN,0.397419,1.0
5,1,"d=3, A→B←C",cd_BOmega_l0_0.2,1.0,1.0,0.0,NaN,0.385381,1.0
6,1,"d=3, A→B←C",cd_B_l0_0.0,1.0,1.0,0.0,NaN,0.321519,1.0
7,1,"d=3, A→B←C",cd_B_l0_0.1,1.0,1.0,0.0,NaN,0.321810,1.0
8,1,"d=3, A→B←C",cd_B_l0_0.2,1.0,1.0,0.0,NaN,0.405576,1.0
9,1,"d=3, A→B←C",ges,1.0,1.0,0.0,NaN,0.019373,1.0


In [ ]:
# 正式实验：每个 experiment 生成 50 次数据
# 如果你想立刻跑完整实验，直接运行本单元

df_trials, df_summary, skip_logs = run_manual_dag_benchmark(
    cfg=CFG,
    n_repeats=CFG['n_repeats'],
    save_tag=CFG['tag'],
)

print('Full trials shape:', df_trials.shape)
print('Skip count:', len(skip_logs))
display(df_summary)

# 额外：给出跨 9 个 experiments 的算法总体平均（便于整体对比）
df_overall = (
    df_trials.groupby('algorithm', as_index=False)
    .agg(
        mec_match_mean=('mec_match', 'mean'),
        exact_match_mean=('exact_match', 'mean'),
        cpdag_shd_mean=('cpdag_shd', 'mean'),
        cpdag_shd_std=('cpdag_shd', 'std'),
        runtime_sec_mean=('runtime_sec', 'mean'),
        runs=('algorithm', 'size'),
    )
)

# 按固定算法顺序展示
alg_order_map = {a: i for i, a in enumerate(ALGORITHM_ORDER)}
df_overall['__order'] = df_overall['algorithm'].map(alg_order_map).fillna(999)
df_overall = df_overall.sort_values('__order').drop(columns='__order').reset_index(drop=True)

overall_path = os.path.join(CFG['out_dir'], f"{CFG['tag']}_overall_summary.csv")
df_overall.to_csv(overall_path, index=False)
print('Overall summary saved:', overall_path)
display(df_overall)


[Experiment 1/9] d=3, A→B←C done
[Experiment 2/9] d=3, A→B→C done
[Experiment 3/9] d=3, A→B→C + A→C done
